In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os

# ── Page config ────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="House Price Predictor",
    page_icon="🏠",
    layout="centered",
)

# ── Load models ─────────────────────────────────────────────────────────────────
MODEL_DIR = os.path.abspath("models")

@st.cache_resource
def load_models():
    models = {}
    paths = {
        "Linear Regression": os.path.join(MODEL_DIR, "linear_reg_pipeline.pkl"),
        "SVR":               os.path.join(MODEL_DIR, "svr_pipeline.pkl"),
        "Random Forest":     os.path.join(MODEL_DIR, "rf_pipeline.pkl"),
    }
    for name, path in paths.items():
        if os.path.exists(path):
            models[name] = joblib.load(path)
    return models

models = load_models()

# ── Header ──────────────────────────────────────────────────────────────────────
st.title("🏠 House Price Predictor")
st.caption("Ames Housing Dataset · Enter property details to estimate the sale price.")
st.divider()

# ── Sidebar – model selector ────────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Settings")
    if models:
        selected_model_name = st.selectbox("Model", list(models.keys()))
        selected_model = models[selected_model_name]
    else:
        st.warning("No trained models found.\nMake sure the `../models/` folder contains the `.pkl` files.")
        selected_model = None

    st.divider()
    st.markdown(
        "**Features used:** 18 original → engineered & reduced to 14 "
        "before the model sees them.\n\n"
        "**Target:** `SalePrice` (log-transformed during training, "
        "inverse-transformed for display)."
    )

# ── Input form ──────────────────────────────────────────────────────────────────
st.subheader("Property Details")

col1, col2 = st.columns(2)

with col1:
    st.markdown("**Location & Configuration**")
    LotConfig = st.selectbox(
        "Lot Configuration",
        ["Inside", "Corner", "CulDSac", "FR2", "FR3"],
        help="General shape of the lot.",
    )
    Neighborhood = st.selectbox(
        "Neighborhood",
        sorted([
            "Blmngtn", "Blueste", "BrDale", "BrkSide", "ClearCr", "CollgCr",
            "Crawfor", "Edwards", "Gilbert", "IDOTRR", "MeadowV", "Mitchel",
            "NAmes",   "NoRidge", "NPkVill", "NridgHt", "NWAmes",  "OldTown",
            "SWISU",   "Sawyer",  "SawyerW", "Somerst", "StoneBr", "Timber",
            "Veenker",
        ]),
        index=12,
        help="Physical location within Ames city limits.",
    )

    st.markdown("**Quality Ratings**")
    OverallQual = st.slider("Overall Quality", 1, 10, 5,
                            help="Overall material and finish quality (1 = Very Poor, 10 = Very Excellent).")
    KitchenQual = st.select_slider(
        "Kitchen Quality",
        options=["Po", "Fa", "TA", "Gd", "Ex"],
        value="TA",
        format_func=lambda x: {"Po": "Poor", "Fa": "Fair", "TA": "Average",
                                "Gd": "Good", "Ex": "Excellent"}[x],
    )
    ExterQual = st.select_slider(
        "Exterior Quality",
        options=["Po", "Fa", "TA", "Gd", "Ex"],
        value="TA",
        format_func=lambda x: {"Po": "Poor", "Fa": "Fair", "TA": "Average",
                                "Gd": "Good", "Ex": "Excellent"}[x],
    )
    BsmtQual = st.select_slider(
        "Basement Quality",
        options=["NA", "Po", "Fa", "TA", "Gd", "Ex"],
        value="TA",
        format_func=lambda x: {"NA": "No Basement", "Po": "Poor", "Fa": "Fair",
                                "TA": "Average", "Gd": "Good", "Ex": "Excellent"}[x],
    )

with col2:
    st.markdown("**Size (sq ft)**")
    GrLivArea   = st.number_input("Above-Ground Living Area (sq ft)", 300,  5000, 1500, step=50)
    TotalBsmtSF = st.number_input("Total Basement Area (sq ft)",       0,    3500, 800,  step=50)
    FirstFlrSF  = st.number_input("1st Floor Area (sq ft)",            300,  4000, 900,  step=50)
    SecondFlrSF = st.number_input("2nd Floor Area (sq ft)",            0,    2000, 0,    step=50,
                                  help="Enter 0 if no second floor.")

    st.markdown("**Year**")
    YearBuilt    = st.number_input("Year Built",       1870, 2010, 1980)
    YearRemodAdd = st.number_input("Year Remodelled",  1950, 2010, 1995,
                                   help="Same as Year Built if no remodel.")

    st.markdown("**Rooms & Garage**")
    FullBath     = st.number_input("Full Bathrooms",          0, 5, 2)
    BsmtFullBath = st.number_input("Basement Full Bathrooms", 0, 5, 0)
    HalfBath     = st.number_input("Half Bathrooms",          0, 3, 0)
    BsmtHalfBath = st.number_input("Basement Half Bathrooms", 0, 3, 0)
    TotRmsAbvGrd = st.number_input("Total Rooms Above Ground", 1, 15, 6)
    GarageCars   = st.number_input("Garage Capacity (cars)",   0, 5,  2)

st.divider()

# ── Predict ─────────────────────────────────────────────────────────────────────
predict_btn = st.button("Predict Sale Price", type="primary", use_container_width=True)

if predict_btn:
    if selected_model is None:
        st.error("No model loaded. Check the `../models/` folder.")
    else:
        # Build a one-row DataFrame with all 18 original features
        input_df = pd.DataFrame([{
            "LotConfig":    LotConfig,
            "Neighborhood": Neighborhood,
            "KitchenQual":  KitchenQual,
            "OverallQual":  int(OverallQual),
            "2ndFlrSF":     int(SecondFlrSF),
            "1stFlrSF":     int(FirstFlrSF),
            "TotalBsmtSF":  int(TotalBsmtSF),
            "YearBuilt":    int(YearBuilt),
            "YearRemodAdd": int(YearRemodAdd),
            "ExterQual":    ExterQual,
            "BsmtQual":     BsmtQual if BsmtQual != "NA" else None,
            "FullBath":     int(FullBath),
            "BsmtFullBath": int(BsmtFullBath),
            "HalfBath":     int(HalfBath),
            "BsmtHalfBath": int(BsmtHalfBath),
            "GrLivArea":    int(GrLivArea),
            "TotRmsAbvGrd": int(TotRmsAbvGrd),
            "GarageCars":   int(GarageCars),
        }])

        try:
            log_pred   = selected_model.predict(input_df)[0]
            price_pred = np.expm1(log_pred)

            st.success("Prediction complete!")
            st.metric(
                label=f"Estimated Sale Price  ({selected_model_name})",
                value=f"${price_pred:,.0f}",
            )

            # Confidence band (±10 % rough heuristic)
            low  = price_pred * 0.90
            high = price_pred * 1.10
            st.caption(f"Approximate range: **${low:,.0f}** – **${high:,.0f}**  (±10 %)")

        except Exception as e:
            st.error(f"Prediction failed: {e}")

# ── Footer ──────────────────────────────────────────────────────────────────────
st.divider()
st.caption("House Price Prediction Project · Ames, Iowa Housing Dataset (Kaggle)")

2026-06-26 17:29:57.366 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 17:29:57.368 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 17:29:57.369 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 17:29:57.948 
  command:

    streamlit run c:\Jupiter\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-06-26 17:29:57.950 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 17:29:57.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-26 17:29:57.953 Thread 'MainThread': missin

DeltaGenerator()